In [1]:
# Финальное решение - результат 295.85155
import pandas as pd
import numpy as np
import gdown
from sklearn.preprocessing import RobustScaler
from lightgbm import LGBMRegressor
from scipy.stats import boxcox
from scipy.special import inv_boxcox
import warnings
warnings.filterwarnings('ignore')

# ID файлов
train_id = '159PZX3X5rpUO-WbzWyC9whnc8B4mNqJl'
test_id = '1Ui2t87X3in-Wu-pnjkDXa_VtPsVafi0l'
sample_id = '1LL6moSzpUVxJUTMeXihWvUxBJNjvj6EH'

# Скачиваем
gdown.download(f'https://drive.google.com/uc?id={train_id}', 'train.csv', quiet=False)
gdown.download(f'https://drive.google.com/uc?id={test_id}', 'test.csv', quiet=False)
gdown.download(f'https://drive.google.com/uc?id={sample_id}', 'sample_submission.csv', quiet=False)

# Читаем
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')
df_sample = pd.read_csv('sample_submission.csv')

# Заполняем пропуски
null_cols = ['MaxPartialCharge', 'MinPartialCharge', 'MaxAbsPartialCharge',
             'MinAbsPartialCharge', 'BCUT2D_MWHI', 'BCUT2D_MWLOW',
             'BCUT2D_CHGHI', 'BCUT2D_CHGLO', 'BCUT2D_LOGPHI',
             'BCUT2D_LOGPLOW', 'BCUT2D_MRHI', 'BCUT2D_MRLOW']

for col in null_cols:
    train_median = df_train[col].median()
    df_train[col] = df_train[col].fillna(train_median)
    df_test[col] = df_test[col].fillna(train_median)

# Лучшие параметры
best_params = {
    'IC50, mM': {'num_leaves': 31, 'min_child_samples': 20, 'learning_rate': 0.05},
    'CC50, mM': {'num_leaves': 15, 'min_child_samples': 10, 'learning_rate': 0.1},
    'SI': {'num_leaves': 31, 'min_child_samples': 20, 'learning_rate': 0.05}
}

targets = ['IC50, mM', 'CC50, mM', 'SI']
feature_cols = [col for col in df_train.columns if col not in ['index', 'IC50, mM', 'CC50, mM', 'SI']]

predictions = {}

for target in targets:
    print(f"{target}")

    X_full = df_train[feature_cols].copy()
    y_full = df_train[target].copy()
    X_test = df_test[feature_cols].copy()

    # Box-Cox вместо log1p
    y_shifted = y_full + 1e-8
    y_transformed, lambda_ = boxcox(y_shifted)

    scaler = RobustScaler()
    X_full_scaled = scaler.fit_transform(X_full)
    X_test_scaled = scaler.transform(X_test)

    model = LGBMRegressor(**best_params[target], random_state=42, verbosity=-1)
    model.fit(X_full_scaled, y_transformed)

    pred_transformed = model.predict(X_test_scaled)
    predictions[target] = inv_boxcox(pred_transformed, lambda_)

# Сабмит
submission = pd.DataFrame({
    'index': df_sample['index'],
    'IC50': predictions['IC50, mM'],
    'CC50': predictions['CC50, mM'],
    'SI': predictions['SI']
})

submission.to_csv('submission_boxcox.csv', index=False)
print("\nsubmission_boxcox.csv сохранён")

Downloading...
From: https://drive.google.com/uc?id=159PZX3X5rpUO-WbzWyC9whnc8B4mNqJl
To: /content/train.csv
100%|██████████| 1.36M/1.36M [00:00<00:00, 65.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Ui2t87X3in-Wu-pnjkDXa_VtPsVafi0l
To: /content/test.csv
100%|██████████| 441k/441k [00:00<00:00, 40.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1LL6moSzpUVxJUTMeXihWvUxBJNjvj6EH
To: /content/sample_submission.csv
100%|██████████| 15.4k/15.4k [00:00<00:00, 33.0MB/s]


IC50, mM
CC50, mM
SI

submission_boxcox.csv сохранён
